# 可变形注意力

> 目标：把“标准注意力为什么贵、可变形注意力到底改了什么、输入里每个元素代表什么、代码里每一步在做什么”串成一条连续逻辑。

> 这份笔记默认你已经见过 Transformer，但不假设你已经理解 Deformable DETR 或 BEVFormer。

> 阅读路线：
1. 先回顾标准注意力里的 query、key、value 到底是什么意思
2. 再看视觉任务里为什么不想让每个 query 和整张特征图全量匹配
3. 然后引入卷积与可变形卷积，理解“规则采样”到“可学习采样”的变化
4. 最后再看可变形注意力：参考点、偏移量、采样权重分别承担什么角色
5. 对照 PyTorch 代码，看这些概念如何落到张量实现上

> 一句话先说结论：
- 标准注意力是“先和所有位置算相关性，再决定关注谁”。
- 可变形注意力是“直接预测应该去哪些位置采样，再把采样结果加权融合”。

> 因此它不是在标准注意力上做一个小修小补，而是把“全量匹配”改成了“稀疏、连续、可学习采样”。

## 1. 可变形注意力和可变形卷积的关系

两者的共同思想都是：
- 不在固定规则位置取样。
- 让网络动态决定去哪里取信息。

但两者也有明显区别：
- 可变形卷积通常是围绕局部邻域做可学习采样。
- 可变形注意力则是围绕一个参考点，在更大范围内做稀疏采样与加权融合。

你可以先把可变形注意力理解成：
- 保留了“动态采样位置”这个思想。
- 同时引入了注意力里的“内容相关加权融合”能力。

## 2. 从标准注意力一步步走到可变形注意力

> 这一节只做一件事：把“标准注意力”里的元素，平滑地映射到“可变形注意力”里。

### 2.1 标准注意力里，query、key、value 分别是什么

设有一组输入特征 $x$，经过线性映射后得到：
- $Q = xW_Q$
- $K = xW_K$
- $V = xW_V$

对某个 query $q_i$，标准注意力做的是：

$
\mathrm{Attn}(q_i) = \sum_j \alpha_{ij} v_j, \qquad
\alpha_{ij} = \mathrm{softmax}\left( \frac{q_i k_j^T}{\sqrt{d}} \right)
$

这里每个元素的含义是：
- `query`：当前我想更新的那个位置或 token。
- `key`：候选信息位置的“索引签名”，用来和 query 算相似度。
- `value`：真正要被取回并聚合的信息。

所以标准注意力本质上是：
- 先拿 query 和所有 key 做匹配。
- 再用匹配分数去加权所有 value。

### 2.2 放到视觉特征图上，会发生什么

如果把一张特征图展平成 $N=H \times W$ 个位置：
- 每个空间位置都可以看成一个 token。
- 每个 query 可能都要和全部 $N$ 个 key 交互。
- 如果 query 数也很多，那么相关性矩阵就会很大。

这就是视觉里标准全局注意力昂贵的根源：
- 不是聚合本身太难。
- 而是“先和所有位置比较一遍”这一步太贵。

### 2.3 一个关键想法：真的需要和所有 key 比吗

在很多视觉任务中，答案通常是否定的。
- 一个目标只和图像中少量区域强相关。
- 一个 BEV 网格点也只会对应少量视角中的局部区域。
- 大量无关位置其实只是陪跑。

于是就有一个自然问题：
- 能不能不要先和所有 key 做匹配？
- 能不能直接预测“应该去哪里看”？

可变形注意力给出的答案就是：可以。

### 2.4 从“全量匹配”变成“稀疏采样”

标准注意力的流程是：
1. 对全部 key 位置做相似度计算。
2. 用 softmax 得到全局权重。
3. 对全部 value 做加权和。

可变形注意力把它改成：
1. 先给每个 query 一个参考点 `reference_point`。
2. 再预测少量采样偏移 `offsets`。
3. 在这些采样位置上，从 value 特征图中取值。
4. 对这些采样值做加权求和。

这意味着：
- 不再显式构造“query 对所有 key”的大矩阵。
- key 不再以“离散候选集合”的方式出现。
- 取而代之的是“连续坐标上的采样位置”。

### 2.5 那 key 去哪了

这是很多人第一次看可变形注意力时最困惑的点。

一个容易理解的说法是：
- 在标准注意力里，key 的职责是“告诉 query 哪些位置相关”。
- 在可变形注意力里，这个职责被“参考点 + 偏移量 + 采样权重”部分接管了。

也就是说，在很多实现里：
- 不再显式地构造一整张 `K` 去和 `Q` 做全量点积。
- 而是由 query 直接预测出该去哪里采样，以及每个采样点该占多大权重。

所以你可以把可变形注意力理解为：
- `query` 仍然存在，而且是核心驱动信号。
- `value` 仍然存在，因为最终被聚合的信息还是来自特征图。
- `key` 的“全局离散匹配”角色被弱化或隐式化，变成了“直接预测采样策略”。

In [ ]:
import torch
import torch.nn as nn

# 固定随机种子，确保每次运行结果一致，方便调试
torch.manual_seed(0)

# ------------------------------------------------------------
# 1. 构造输入：一个特征图（模拟视觉任务中的 feature map）
# ------------------------------------------------------------
# feature_map 形状: (batch_size=1, height=4, width=4, channels=8)
# 表示一张 4×4 的空间特征图，每个位置有 8 维特征向量
feature_map = torch.randn(1, 4, 4, 8)

# 将空间维度（高、宽）展平，得到 16 个 token（每个 token 是 8 维向量）
# flattened_tokens 形状: (1, 16, 8) —— 1 个 batch，16 个位置，每个位置 8 维
flattened_tokens = feature_map.view(1, 16, 8)

# ------------------------------------------------------------
# 2. 定义 Query、Key、Value（这里为了演示，三者共享同一组数据）
# ------------------------------------------------------------
# query_token: 假设我们只有一个查询向量（例如当前解码位置的特征）
# 形状: (1, 1, 8) —— 1 个 batch，1 个 query，每个 query 8 维
query_token = torch.randn(1, 1, 8)

# keys = 所有 token 的特征，用于和 query 计算相似度
# 形状: (1, 16, 8)
keys = flattened_tokens

# values = 同样所有 token 的特征，用于加权求和得到最终输出
# 形状: (1, 16, 8)
values = flattened_tokens

# ------------------------------------------------------------
# 3. 计算注意力分数（缩放点积注意力）
# ------------------------------------------------------------
# 计算 query 与所有 key 的点积，得到每个位置的原始分数
# torch.matmul(query_token, keys.transpose(-1, -2))
#   query_token: (1, 1, 8) 与 keys 的转置 (1, 8, 16) 相乘
#   结果形状: (1, 1, 16) —— 表示该 query 对 16 个 key 的未缩放分数
# 然后除以 sqrt(d_k) = sqrt(8) 进行缩放，防止梯度进入 softmax 饱和区
attention_scores = torch.matmul(query_token, keys.transpose(-1, -2)) / (8 ** 0.5)

# ------------------------------------------------------------
# 4. 应用 softmax 得到注意力权重（概率分布）
# ------------------------------------------------------------
# softmax 沿着最后一个维度（即 key 的位置维度）做归一化
# attention_weights 形状: (1, 1, 16) —— 每个位置的权重，和为 1
attention_weights = torch.softmax(attention_scores, dim=-1)

# ------------------------------------------------------------
# 5. 用注意力权重对 values 进行加权求和，得到输出
# ------------------------------------------------------------
# torch.matmul(attention_weights, values)
#   attention_weights: (1, 1, 16) 与 values: (1, 16, 8) 相乘
#   数学含义：对 16 个位置的 8 维向量按权重求和
#   输出形状: (1, 1, 8) —— 最终的聚合特征向量
attention_output = torch.matmul(attention_weights, values)

# ------------------------------------------------------------
# 6. 打印诊断信息，验证计算正确性
# ------------------------------------------------------------
print("全部候选位置数:", keys.shape[1])                     # 应该输出 16
print("attention_weights shape:", attention_weights.shape)   # (1, 1, 16)
# 验证权重之和是否等于 1（由于浮点数误差，可能是 0.9999...）
print("权重和是否为 1:", attention_weights.sum(dim=-1))      # 应该接近 [1.]
print("输出向量 shape:", attention_output.shape)             # (1, 1, 8)

# 找到注意力权重最大的位置（对应最相关的 token 索引）
# 因为 attention_weights 形状为 (batch, num_queries=1, num_keys=16)
# 所以取 [0, 0] 对应的权重向量，找最大值索引
max_idx = attention_weights[0, 0].argmax().item()
print("注意力最大的空间位置索引:", max_idx)                  # 具体数值依赖随机初始化，但可重现

全部候选位置数: 16
attention_weights shape: torch.Size([1, 1, 16])
权重和是否为 1: tensor([[1.]])
输出向量 shape: torch.Size([1, 1, 8])
注意力最大的空间位置索引: 0


## 3. 输入元素到底是什么：和传统 Q、K、V 的对应关系

> 这一节专门解决“模块输入里每个张量到底代表谁”的问题。

为了便于理解，先看单尺度版本。假设我们有：
- `query`: 形状为 $(B, N_q, C)$
- `value`: 形状为 $(B, H, W, C)$
- `reference_points`: 形状为 $(B, N_q, 2)$

它们分别表示：

### 3.1 query：谁要被更新

`query` 对应标准注意力里的 `Q`。
- 它代表当前要更新的那批查询向量。
- 在检测里，它可以是 object query。
- 在 BEVFormer 里，它可以是 BEV query。
- 在更一般的场景里，它就是“我现在想从特征图里取信息来更新谁”。

所以每个 query 都会独立地产生：
- 一个或多个参考点
- 若干个采样偏移
- 若干个采样权重

### 3.2 value：真正被读取的信息源

`value` 对应标准注意力里的 `V`。
- 它通常来自某张特征图或多层特征图。
- 真正被双线性采样、再被加权融合的内容，就是它。

注意这里 `value` 不是展平的 token 序列，而是保留了二维空间结构的特征图。
这很重要，因为：
- 可变形注意力要在连续坐标上采样。
- 所以保留 $(H, W)$ 的几何布局会更自然。

### 3.3 reference_points：粗定位锚点

`reference_points` 不是标准注意力里的原生元素，它是可变形注意力新增的关键变量。
- 它表示这个 query 大概应该去哪里找信息。
- 可以理解成“粗定位”。
- 后续所有偏移采样，都是围绕它展开。

它的来源依任务而不同：
- 在 Deformable DETR 中，常由 query 经过线性层预测并归一化。
- 在 BEVFormer 中，常与 3D 到 2D 的几何投影有关。

### 3.4 sampling_offsets：细粒度搜索方向

`sampling_offsets` 通常由 query 通过线性层预测得到。
- 形状常写成 $(B, N_q, N_h, K, 2)$。
- 表示每个 query、每个 head、每个采样点的二维偏移。

它的作用相当于：
- 参考点先告诉你“中心大概在哪”。
- 偏移量再告诉你“具体往周围哪些位置看”。

### 3.5 attention_weights：对采样结果做融合

`attention_weights` 同样通常由 query 预测得到。
- 形状常写成 $(B, N_q, N_h, K)$。
- 每个 head 的 $K$ 个采样点会经过 softmax 归一化。

它对应的是注意力里“加权融合”的那部分能力。
只是这里被加权的对象，不再是全部 value 位置，而是少量被采样出来的特征值。

### 3.6 用一句对照总结

把标准注意力和可变形注意力并排看：
- 标准注意力：`query` 与全部 `key` 匹配，再聚合全部 `value`。
- 可变形注意力：`query` 直接预测采样策略，在 `value` 上稀疏取样并聚合。

因此新增的核心对象就是：
- `reference_points`：先去哪里附近看
- `sampling_offsets`：具体采哪些点
- `attention_weights`：这些点如何融合

In [2]:
# 观察 reference_points、sampling_offsets、sampling_locations 的形状与数值关系
batch_size = 1
num_query = 2
num_heads = 2
num_points = 4

reference_points_demo = torch.tensor([[[0.50, 0.50], [0.25, 0.75]]])
sampling_offsets_demo = torch.tensor(
    [[[[[1.0, 0.0], [0.0, 1.0], [-1.0, 0.0], [0.0, -1.0]],
       [[2.0, 0.0], [0.0, 2.0], [-2.0, 0.0], [0.0, -2.0]]],
      [[[1.0, 1.0], [-1.0, 1.0], [-1.0, -1.0], [1.0, -1.0]],
       [[0.5, 0.5], [-0.5, 0.5], [-0.5, -0.5], [0.5, -0.5]]]]],
    dtype=torch.float32,
 )

height, width = 20, 30
normalizer = torch.tensor([width, height], dtype=torch.float32).view(1, 1, 1, 1, 2)
sampling_locations_demo = (
    reference_points_demo[:, :, None, None, :] + sampling_offsets_demo / normalizer
)

print("reference_points shape:", reference_points_demo.shape)
print("sampling_offsets shape:", sampling_offsets_demo.shape)
print("sampling_locations shape:", sampling_locations_demo.shape)
print("第 1 个 query、第 1 个 head 的采样位置:\n", sampling_locations_demo[0, 0, 0])

reference_points shape: torch.Size([1, 2, 2])
sampling_offsets shape: torch.Size([1, 2, 2, 4, 2])
sampling_locations shape: torch.Size([1, 2, 2, 4, 2])
第 1 个 query、第 1 个 head 的采样位置:
 tensor([[0.5333, 0.5000],
        [0.5000, 0.5500],
        [0.4667, 0.5000],
        [0.5000, 0.4500]])


好的，我把 `W_m` 的解释整合进 4.2 节：

---

# 4. 数学表达：从标准注意力到可变形注意力

> 核心思路：不急着背公式，先看"公式里哪些部分被替换了"。


## 4.1 标准注意力在做什么

对第 $i$ 个 query，标准注意力公式为：

$$
\mathrm{Attn}(q_i) = \sum_{j=1}^{N} \alpha_{ij} v_j, \qquad
\alpha_{ij} = \mathrm{softmax}\left( \frac{q_i k_j^T}{\sqrt{d}} \right)
$$

**物理含义拆解**：
- **第一步**：用 $q_i$ 与所有 $k_j$ 做点积，计算与全部 $N$ 个位置的相关性。
- **第二步**：用 softmax 归一化成权重，对全部 $N$ 个 $v_j$ 做加权和。

**关键特征**：计算量与 $N$ 成正比。在图像中 $N = H \times W$，当分辨率较高时，这个计算开销非常大。


## 4.2 可变形注意力改写了哪一步？参数怎么生成？

可变形注意力**不再枚举全部 $N$ 个位置**，而是直接把"遍历所有位置"替换成"只访问少量采样点"。

**公式定义**（单尺度版本）：

设：
- query 表示为 $q_i$，共有 $M$ 个 query
- value 特征图为 $x \in \mathbb{R}^{H \times W \times C}$
- 第 $i$ 个 query 的参考点为 $p_i$（归一化坐标，范围 $[0,1]$）
- 第 $m$ 个 head、第 $k$ 个采样点的偏移量为 $\Delta p_{imk}$
- 对应注意力权重为 $A_{imk}$

则单尺度可变形注意力写成：

$$
\mathrm{DeformAttn}(q_i, x) = \sum_{m=1}^{N_h} W_m \left( \sum_{k=1}^{K} A_{imk} \cdot x\big(p_i + \Delta p_{imk}\big) \right)
$$

**公式中关键符号的含义**：

- **$W_m$**：第 $m$ 个 attention head 的**可学习输出投影矩阵**。每个头从特征图上采集并加权聚合后的特征向量经过 $W_m$ 线性变换，被映射到最终的输出空间，实现多头信息的融合。在标准 Transformer 多头注意力中，这相当于 $W_o$（输出投影矩阵）针对每个 head 的切片，形状通常为 $(d_{model}, d_{head})$。

- **$x(p_i + \Delta p_{imk})$**：表示通过浮点数坐标索引特征图，由于坐标可能不在整数网格上，需要用双线性插值采样。

- **$A_{imk}$**：不是对全图所有位置的权重，而**只针对 $K$ 个稀疏采样点**的注意力权重。

- 每个 head 可以独立学习不同的采样模式。

**采样参数是怎么算出来的？**

在实际实现（如 Deformable DETR）中，$A_{imk}$ 和 $\Delta p_{imk}$ **完全由 Query 自身预测生成**，不依赖任何 Key 的内容：

- **采样偏移量**：
  $$
  \Delta p_{imk} = \text{sigmoid}(\text{Linear}_{offset}(q_i)) - 0.5
  $$
  将偏移量限制在 $[-0.5, 0.5]$ 范围内，使采样点围绕参考点 $p_i$ 局部移动。

- **注意力权重**：
  $$
  A_{imk} = \text{Softmax}\left( \text{Linear}_{attn}(q_i) \right)
  $$
  在同一个 head 内的 $K$ 个采样点之间做 softmax 归一化，保证每个 head 的 $K$ 个权重之和为 1。

**关键结论**：这里**没有计算 $QK^T$**，因此计算量与特征图尺寸 $H \times W$ 完全解耦，只与 $K$（采样点数）和 head 数量有关。


## 4.3 为什么这仍然可以称为"注意力"？

虽然没有显式写出 $QK^T$，但它仍然具备注意力的两个核心特征：

| 注意力核心特征 | 可变形注意力如何实现 |
| :--- | :--- |
| **由 query 决定"关注哪里"** | query 通过线性层预测 $\Delta p$，决定采样位置 |
| **对取回的信息做内容相关的加权融合** | query 通过线性层预测 $A$，对不同采样点加权 |

**本质区别在于**：
- **标准注意力**：通过"和全部 key 匹配"来决定关注位置（动态的、内容依赖的）。
- **可变形注意力**：通过"直接预测采样位置和权重"来决定关注位置（动态的、但仅依赖 query，不依赖 key 的内容）。

**直觉上**：标准注意力像一个"全搜索"系统，需要把 query 和所有候选位置做对比；而可变形注意力像一个"专家系统"，query 直接判断"我应该去看哪些位置"。后者的速度远快于前者，但可能牺牲全局上下文感知能力。


## 4.4 多尺度版本再多了什么？

如果是多尺度版本，设第 $l$ 层特征图为 $x^l$，则公式扩展为：

$$
\mathrm{MSDeformAttn}(q_i) = \sum_{m=1}^{N_h} W_m \left( \sum_{l=1}^{L} \sum_{k=1}^{K} A_{imlk} \cdot x^l\big(p_i^l + \Delta p_{imlk}\big) \right)
$$

**相比单尺度只多了一个维度**：
- 从一张特征图采样 → 从 $L$ 张不同分辨率的特征图采样。
- 每层都有自己的采样点偏移量 $\Delta p$ 和权重 $A$。
- 不同层之间共享同一个参考点 $p_i$（即同一个物理位置）。

**重要的多尺度细节**：
$p_i^l$ 表示参考点在第 $l$ 层特征图上的归一化坐标。由于不同尺度的特征图尺寸不同（如 1/4、1/8、1/16 下采样），同一物理位置映射到不同层时，坐标值需要**按该层的 stride 进行归一化**，确保 $p_i^l \in [0,1]$，以便 PyTorch 的 $F.grid\_sample$ 能够正确索引。


## 4.5 复杂度为什么更低？

用直观的规模对比来说明：

| 注意力类型 | 访问位置数 | 计算量级 |
| :--- | :--- | :--- |
| 标准注意力（单尺度） | $H \times W$ | $O(HW)$ |
| 可变形注意力（单尺度） | $N_h \times K$ | $O(N_h \times K)$ |
| 可变形注意力（多尺度） | $N_h \times L \times K$ | $O(N_h \times L \times K)$ |

**当 $K$ 远小于 $H \times W$ 时（通常 $K=4$，$HW$ 可能为 $10^5$），计算量差距显著。**

这也是可变形注意力特别适合高分辨率视觉特征的核心原因——**它把"密集全局匹配"变成了"稀疏局部采样"**。


## 4.6 总结对照表：标准注意力 vs 可变形注意力

| 对比维度 | 标准注意力 | 可变形注意力 |
| :--- | :--- | :--- |
| **空间扫描方式** | 全图密集遍历（$H \times W$ 个位置） | 稀疏定点采样（$K$ 个采样点，$K=4 \sim 8$） |
| **权重来源** | $Q$ 与 $K$ 的点积（依赖 Key 内容） | $Q$ 直接预测 softmax 权重（不依赖 Key） |
| **采样点坐标** | 固定在整数网格上 | 浮点数偏移（由 $Q$ 预测，通过双线性插值取数） |
| **是否依赖 Key 内容** | 是（需要 $Q$ 与所有 $K$ 做点积） | 否（仅用 $Q$ 预测位置和权重） |
| **多尺度处理** | 通常不直接支持 | 天然支持多层特征金字塔采样 |
| **计算复杂度** | $O(HW)$ | $O(N_h \times L \times K)$，与 $HW$ 无关 |
| **适用场景** | NLP 序列、小分辨率图像 | 高分辨率视觉特征（目标检测、分割） |


## 4.7 一句话记忆点

> **可变形注意力 = 用 Query 直接"指路"（预测采样点和权重），而不是"全场搜索"（$QK^T$）。它把密集的全局注意力，简化成了稀疏的局部特征聚合。**

这个"指路"机制使得计算量与图像尺寸解耦，是可变形注意力能在高分辨率特征图上高效运行的根本原因。

In [3]:
# 用一个最小例子看“全量加权”和“稀疏采样加权”的差异
value_map = torch.arange(1, 17, dtype=torch.float32).view(1, 4, 4, 1)

# 标准注意力视角：对全部 16 个位置做加权
all_values = value_map.view(1, 16, 1)
dense_weights = torch.softmax(torch.tensor([[0.2] * 16]), dim=-1).view(1, 16, 1)
dense_output = (dense_weights * all_values).sum(dim=1)

# 可变形注意力视角：只取 4 个采样点做加权
sparse_sample_indices = torch.tensor([5, 6, 9, 10])
sparse_values = all_values[:, sparse_sample_indices]
sparse_weights = torch.softmax(torch.tensor([[1.0, 2.0, 2.0, 1.0]]), dim=-1).view(1, 4, 1)
sparse_output = (sparse_weights * sparse_values).sum(dim=1)

print("标准注意力访问的位置数:", all_values.shape[1])
print("可变形注意力访问的位置数:", sparse_values.shape[1])
print("标准注意力输出:", dense_output.squeeze().item())
print("稀疏采样聚合输出:", sparse_output.squeeze().item())

标准注意力访问的位置数: 16
可变形注意力访问的位置数: 4
标准注意力输出: 8.5
稀疏采样聚合输出: 8.5


## 5. 用一个 query 的视角看一遍完整流程

> 如果你总觉得公式里变量太多，可以只盯住“一个 query”来看。

假设：
- 我们当前只看第 $i$ 个 query。
- 它对应一个向量 $q_i \in \mathbb{R}^{C}$。
- 它的参考点是 $p_i=(0.6, 0.4)$，表示大概去特征图右侧偏上区域找信息。
- 现在有 2 个 head，每个 head 采样 4 个点。

那么这个 query 的工作流程可以写成：

### 第一步：先确定大致看哪里

模型先为这个 query 给出一个参考点 $p_i$。
这个点不是最终采样点，而是一个中心锚点。

### 第二步：每个 head 进一步预测偏移

例如某个 head 预测出 4 个偏移：
- $(-0.03, 0.01)$
- $(0.02, -0.04)$
- $(0.00, 0.05)$
- $(0.04, 0.02)$

于是最终采样位置就是：
- $p_i + \Delta p_{i11}$
- $p_i + \Delta p_{i12}$
- $p_i + \Delta p_{i13}$
- $p_i + \Delta p_{i14}$

这些位置通常是浮点坐标，不会正好落在整数像素中心上。

### 第三步：在 value 特征图上取值

因为采样位置是连续坐标，所以不能直接用整数索引。
通常会用双线性插值：
- 找到附近的 4 个离散网格点
- 按距离做加权
- 得到该连续位置的特征向量

这一步就是代码里的 `grid_sample` 在做的事。

### 第四步：对采样到的若干特征做加权融合

假设这个 head 对 4 个采样点预测出的权重是：

$
[0.1, 0.2, 0.5, 0.2]
$

那么这个 head 的输出就是：

$
0.1 v_1 + 0.2 v_2 + 0.5 v_3 + 0.2 v_4
$

其中 $v_1, v_2, v_3, v_4$ 是 4 个采样位置通过双线性插值得到的特征向量。

### 第五步：多个 head 的结果再拼起来

每个 head 都会做一遍“预测偏移 -> 采样 -> 加权求和”。
最后把所有 head 的结果拼接起来，再过一个线性层，得到这个 query 的更新结果。

所以整个过程本质上是：
- query 不再对全图搜索。
- query 直接决定采样策略。
- value 特征图作为被读取的信息源。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


def _to_grid_sample_format(data_4d: torch.Tensor, num_heads: int, head_dim: int):
    """
    将 (B, H, W, num_heads * head_dim) 格式的特征图转换为 grid_sample 所需的格式。
    
    返回:
        feat_for_grid: (B * num_heads, head_dim, H, W)
    """
    B, H, W, _ = data_4d.shape
    # (B, H, W, num_heads, head_dim) -> (B, num_heads, head_dim, H, W)
    feat = data_4d.view(B, H, W, num_heads, head_dim)
    feat = feat.permute(0, 3, 4, 1, 2).contiguous()
    # 合并 Batch 和 Head 维度，让 grid_sample 能并行处理所有 Head
    return feat.view(B * num_heads, head_dim, H, W)


class SimplifiedDeformableAttention(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, num_points: int):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim 必须能被 num_heads 整除"

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.num_points = num_points
        self.head_dim = embed_dim // num_heads

        # 1. 对 Value 做线性投影
        self.value_proj = nn.Linear(embed_dim, embed_dim)
        
        # 2. 由 Query 预测采样偏移量 (Offset) 和 调制权重 (Modulation)
        self.offset_proj = nn.Linear(embed_dim, num_heads * num_points * 2)
        self.modulation_proj = nn.Linear(embed_dim, num_heads * num_points)
        
        # 3. 多头融合输出
        self.output_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, query, value, reference_points):
        """
        query: (B, Nq, C)               -- 标准 Q
        value: (B, H, W, C)             -- 标准 V (特征图)
        reference_points: (B, Nq, 2)    -- 归一化的锚点坐标 [0, 1]
        """
        B, Nq, C = query.shape
        B, H, W, _ = value.shape

        # ----------------------------------------------------------------
        # Step 1: 特征准备 (将 Value 投影并调整内存布局)
        # ----------------------------------------------------------------
        # 先将 Value 映射到多头空间 (B, H, W, num_heads * head_dim)
        value_proj = self.value_proj(value)
        # 转换为 grid_sample 需要的输入格式: (B*num_heads, head_dim, H, W)
        # 这样 batch 里的每张图和每个头都是独立的采样样本
        value_for_grid = _to_grid_sample_format(value_proj, self.num_heads, self.head_dim)

        # ----------------------------------------------------------------
        # Step 2: 由 Query 预测采样指令 (即偏移量和权重)
        # ----------------------------------------------------------------
        # 2.1 预测偏移: (B, Nq, num_heads * num_points * 2)
        offsets_raw = self.offset_proj(query)
        offsets = offsets_raw.view(B, Nq, self.num_heads, self.num_points, 2)

        # 2.2 预测调制权重: (B, Nq, num_heads * num_points)
        mod_raw = self.modulation_proj(query)
        modulation = mod_raw.view(B, Nq, self.num_heads, self.num_points)
        # 在同一 head 的 num_points 之间做 Softmax，保证每个 Head 的权重和为 1
        modulation = F.softmax(modulation, dim=-1)  # (B, Nq, num_heads, num_points)

        # ----------------------------------------------------------------
        # Step 3: 计算采样点的最终坐标 (归一化坐标系 [0, 1])
        # ----------------------------------------------------------------
        # 注意: offsets 是像素单位 (比如偏移 2.3 个像素)，需要除以特征图尺寸转为归一化坐标
        # 参考点坐标 range [0, 1]，加上归一化偏移后依然是 [0, 1]
        normalizer = query.new_tensor([W, H]).view(1, 1, 1, 1, 2)  # (1,1,1,1,2)
        # 防止除零 (虽然实际图像很少为0，但加个 clamp 是个好习惯)
        normalizer = normalizer.clamp(min=1.0)
        
        # reference_points[:, :, None, None, :] 扩展为 (B, Nq, 1, 1, 2) 以广播
        sampling_locations = reference_points[:, :, None, None, :] + (offsets / normalizer)

        # ----------------------------------------------------------------
        # Step 4: 转换为 grid_sample 需要的格式 ([-1, 1] 坐标系)
        # ----------------------------------------------------------------
        # 将 [0,1] -> [-1,1] 映射
        grid = sampling_locations * 2.0 - 1.0  # (B, Nq, num_heads, num_points, 2)
        
        # 调整维度以适配 grid_sample: (B, num_heads, Nq, num_points, 2)
        # 这里 grid_sample 的 H_out = Nq, W_out = num_points
        grid = grid.permute(0, 2, 1, 3, 4).contiguous()
        grid = grid.view(B * self.num_heads, Nq, self.num_points, 2)

        # ----------------------------------------------------------------
        # Step 5: 执行双线性插值采样
        # ----------------------------------------------------------------
        # value_for_grid: (B*num_heads, head_dim, H, W)
        # grid:          (B*num_heads, Nq, num_points, 2)
        # 输出:          (B*num_heads, head_dim, Nq, num_points)
        sampled = F.grid_sample(
            value_for_grid,
            grid,
            mode="bilinear",
            padding_mode="zeros",
            align_corners=False,
        )

        # ----------------------------------------------------------------
        # Step 6: 形状恢复为多头的标准格式
        # ----------------------------------------------------------------
        # (B*num_heads, head_dim, Nq, num_points) 
        # -> (B, num_heads, head_dim, Nq, num_points)
        sampled = sampled.view(B, self.num_heads, self.head_dim, Nq, self.num_points)

        # 调整为 (B, Nq, num_heads, num_points, head_dim)
        # 方便后面做加权求和 (将 num_points 维求和消去)
        sampled = sampled.permute(0, 3, 1, 4, 2).contiguous()

        # ----------------------------------------------------------------
        # Step 7: 加权求和 ( Modulation 点乘并聚合 )
        # ----------------------------------------------------------------
        # modulation: (B, Nq, num_heads, num_points, 1)
        # sampled:    (B, Nq, num_heads, num_points, head_dim)
        # 对 num_points 维度做元素乘后求和，得到 (B, Nq, num_heads, head_dim)
        output = (sampled * modulation.unsqueeze(-1)).sum(dim=3)

        # 将多头特征拼接: (B, Nq, num_heads * head_dim) -> (B, Nq, embed_dim)
        output = output.view(B, Nq, self.embed_dim)

        # 最后的输出投影
        output = self.output_proj(output)

        return output

总结起来，就是由 query 直接预测采样偏移和采样权重，value 投影到多头注意力所需的通道空间，作为被查询的二维图，query得到的偏移量加上传入的参考点得到绝对位置坐标，然后进行归一化，再使用value构建的图与query的绝对归一化坐标双线性插值，得到num_point个实际用于加权求和的value，最后有attention_weight和num_points个value进行加权得到最终输出，输出与query形状一致。

In [5]:
batch_size = 2
num_query = 6
height, width = 20, 30
embed_dim = 64
num_heads = 8
num_points = 4

torch.manual_seed(0)

module = SimplifiedDeformableAttention(
    embed_dim=embed_dim,
    num_heads=num_heads,
    num_points=num_points,
)

query = torch.randn(batch_size, num_query, embed_dim)
value = torch.randn(batch_size, height, width, embed_dim)
reference_points = torch.rand(batch_size, num_query, 2)

output = module(query, value, reference_points)
print("query shape:", query.shape)
print("value shape:", value.shape)
print("reference_points shape:", reference_points.shape)
print("output shape:", output.shape)

query shape: torch.Size([2, 6, 64])
value shape: torch.Size([2, 20, 30, 64])
reference_points shape: torch.Size([2, 6, 2])
output shape: torch.Size([2, 6, 64])



# 6. 代码实现逐段解释：把概念和代码一一对上

> **阅读提示**：这一版代码是教学版，不是官方高性能 CUDA 版本，但它包含了可变形注意力最核心的数据流和控制逻辑。


### 6.1 为什么代码里几乎看不到显式的 `key`

这段代码里你会发现：
- 有 `query`
- 有 `value`
- 但**没有**单独写一个 `key` 张量。

**原因不是 `key` 不重要，而是“谁来找 key”的逻辑变了。**

- **标准注意力**：`query` 和 **所有 `key`** 做矩阵乘法，计算相似度，决定关注哪里。`key` 的作用是“提供被匹配的内容”。
- **可变形注意力**：`query` 直接预测采样位置，**不需要先去跟全体 `key` 做匹配**。原本存储在 `key` 里的“内容相关性”，被 `offset_proj` 和 `weight_proj` 这两个可学习网络替代了。

> **直觉理解**：标准注意力像“海选”——查遍所有人（key）才知道谁相关；可变形注意力像“内部推荐”——直接根据经验（query）告诉你该去找哪几个人。

> **补充说明**：在 Deformable DETR 的完整实现中，`query` 通常是 Object Query 或解码器的输出，而 `value` 来自编码器输出的多尺度特征图。`value` 在空间上是有结构的（H, W），而 `key` 在标准的 Cross-Attention 中虽然提供了“内容”，但在这里我们更依赖 `query` 直接预测几何位置和权重，因此省略了显式的 `key` 投影。

### 6.2 `value_proj` 在做什么？

`value_proj` 对 `value` 做线性映射：
- 输入是空间特征图（形状 `B, H, W, C`）
- 输出通道被投影到当前注意力模块所用的嵌入空间（`embed_dim`）。

随后代码把它 reshape 成多头形式（`B, H, W, num_heads, head_dim`）。

**这一步对应的是**：
- 不同 head 各自处理一部分通道。
- 每个 head 后续会学习**不同的采样模式和权重分布**。

### 6.3 `offset_proj` 和 `modulation_proj` 在做什么？

这两个线性层**只吃 `query`**：

- `offset_proj(query)` → 预测采样偏移量 `(Δx, Δy)`，对应公式里的 `Δp`。
- `modulation_proj(query)` → 预测采样权重（经过 Softmax），对应公式里的 `A`。

这正是“**由 query 决定采样策略**”的代码体现：

> query 不再先去和所有 key 打分，而是直接输出“去哪采、采几点、每个点占多少比重”。

**一个容易忽略的细节**：这里的 `modulation_proj` 输出的权重是在 `num_points` 维度上做 Softmax，意味着**每个 head 内部的 K 个采样点的权重之和为 1**。这与标准注意力中每个 query 对所有 key 的权重和为 1 是遥相呼应的。

### 6.4 为什么要除以 `[width, height]`？

代码里有一步：

```python
normalizer = query.new_tensor([width, height]).view(1, 1, 1, 1, 2)
sampling_locations = reference_points[:, :, None, None, :] + sampling_offsets / normalizer
```

**原因在于坐标参考系不同：**

- `reference_points` 用的是**归一化坐标**，范围在 `[0, 1]` 之间（0 代表最左/上边缘，1 代表最右/下边缘）。
- 而 `offset_proj` 预测的偏移量是**像素尺度**的（比如偏移 `2.3` 个像素）。

如果不做归一化，直接把像素偏移加到归一化坐标上，会导致采样点直接飞出边界。所以需要先把像素偏移**除以特征图的宽高**，缩放到 `[0,1]` 参考系，再与参考点相加。

> **直觉理解**：参考点是“百分比位置”（比如图片的 50% 位置），偏移量是“实际米数”（比如往右走 2 米）。要把米数换算成百分比，就得除以总宽度。

### 6.5 `grid_sample` 为什么是关键？

`grid_sample` 的作用可以概括成一句话：

> 给我一组连续坐标，我帮你从二维特征图上做**双线性插值**取值。

这一步非常关键，因为采样点通常不是整数网格点：
- 如果只能落在整数像素上，偏移量的学习就变成“离散跳跃”，无法梯度回传。
- 有了双线性插值，偏移量可以是小数（如 2.3 像素），网络可以通过梯度调整这个小数，实现**亚像素级别的精确定位**。

> **与前面 DCN 代码的呼应**：你在 `ToyDeformConv2d` 里见过的 `sample_deformable_features` 函数，内部调用的正是 `F.grid_sample`。两者的数学本质完全相同——都是用连续坐标去特征图上“抠”数据。区别仅在于：
> - DCN 里，采样坐标是 `(B, K*K, H, W, 2)`，空间维度是输出特征图。
> - 这里，采样坐标是 `(B*num_heads, Nq, num_points, 2)`，空间维度是 Query 数量和采样点数。

### 6.6 最后那步加权求和，对应的是哪部分注意力？

代码最后：

```python
output = (sampled_value * modulation.unsqueeze(-1)).sum(dim=3)   # dim=3 是 num_points 维
```

这一步就是注意力里的“**聚合**”。

把它和标准注意力对照：

| 操作 | 标准注意力 | 可变形注意力（本代码） |
| :--- | :--- | :--- |
| **对谁聚合** | 对**全部 N 个** value 做加权和 | 对**每个 head 内 K 个**采样点的 value 做加权和 |
| **权重来源** | `softmax(QK^T)` 算出的全局相似度 | `modulation_proj(query)` 直接预测 |
| **空间范围** | 全图 | 参考点周围的稀疏点（由 offsets 决定） |

所以，虽然实现路径不同，但最后一步的数学形式完全一致：**加权求和**。

### 6.7 这份简化代码省略了什么？

为了教学清晰，这里刻意省略了一些工程细节：

- **多尺度（Multi-scale）**：没有实现多层级特征图的采样（标准 Deformable DETR 会从 4 层 FPN 特征中采样）。
- **迭代参考点（Iterative Reference）**：没有实现解码器多层之间参考点传递和精细化。
- **高效的 CUDA Kernel**：实际部署时，`grid_sample` 的频繁调用会有性能开销，官方实现用了定制的 CUDA 算子。
- **Mask 和 Padding 处理**：没有处理特征图中无效区域（如 padding 部分）的采样屏蔽。

但对理解原理来说，它已经保留了最核心的 4 件事：
1. **query 驱动采样策略**（`offset_proj` + `modulation_proj`）
2. **reference point 提供粗定位**（锚点）
3. **offsets 提供细粒度采样**（亚像素偏移）
4. **sampled value 再经过 attention weights 融合**（加权求和）

> **学习建议**：如果你现在去读 Deformable DETR 的官方源码，带着这个“简化版骨架”去对照，你会发现官方代码只是在以下三个方面做了扩展：
> - 多尺度特征图循环
> - 参考点逐层更新
> - 采样点数量（K）调优
>
> 核心的“预测偏移 → grid_sample → 加权求和”这个主干，和这份代码完全一致。这也是为什么我建议你先把这个简化版完全吃透——它包含了可变形注意力最本质的计算图。

## 7. 和 BEVFormer 中用法的关系

> 当你把前面的概念吃透，再看 BEVFormer，就不会觉得“为什么又冒出参考点和采样点”。

可以这样理解：
- 每个 BEV query 对应鸟瞰图上的一个网格位置。
- 这个位置通过相机外参与内参，可以投影到各个相机视角图像上。
- 投影结果提供了一个很自然的参考点。
- 接着模型再围绕这个参考点学习少量偏移做细粒度采样。
- 来自不同相机、不同尺度的采样结果再融合回这个 BEV query。

所以 BEVFormer 中的空间交叉注意力，本质上就是：
- 几何投影先给一个粗定位。
- 可变形注意力再做内容相关的稀疏采样。
- 最后把多视角信息聚合回 BEV 空间。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List, Optional


# ============================================================================
# 第一部分：通用多尺度可变形注意力（核心算法类）
# ============================================================================
class MultiScaleDeformableAttention(nn.Module):
    """
    通用多尺度可变形注意力模块 (Multi-Scale Deformable Attention)

    这是 Deformable DETR 的核心组件。相比于标准 Transformer 注意力（全图密集匹配），
    它让 Query 直接预测“去哪采”（偏移量）和“采多重要”（权重），
    将计算复杂度从 O(H*W) 降为 O(num_points * num_levels)。

    本实现支持两种模式：
    1. 标准模式：传入原始特征图 (B, H, W, C)，内部自动做 Value 投影。
    2. 预投影模式 (already_projected=True)：传入已经投影好的多头特征 (B, num_heads, head_dim, H, W)，
       用于外部已经做过 Value 投影优化的场景（如 BEV 融合）。
    """
    def __init__(
        self,
        embed_dim: int,       # 输入输出总特征维度 (如 256)
        num_heads: int,       # 多头注意力的头数 (如 8)
        num_points: int,      # 每个头在每个尺度上的采样点数 (如 4)。通常 4~8 就足够。
        num_levels: int,      # 多尺度特征金字塔的层数 (如 4)
        dropout: float = 0.1,
    ):
        super().__init__()
        # 校验：总维度必须能被头数整除，确保每个头分到整数维度
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.num_points = num_points
        self.num_levels = num_levels
        self.head_dim = embed_dim // num_heads  # 每个头的特征维度 (如 32)

        # ------------------------------------------------------------
        # 1. Value 投影层 (标准模式使用)
        #    将输入特征图的通道从 embed_dim 映射到 embed_dim。
        #    注意：如果 already_projected=True，该层将被跳过，不消耗计算。
        # ------------------------------------------------------------
        self.value_proj = nn.Linear(embed_dim, embed_dim)

        # ------------------------------------------------------------
        # 2. 由 Query 驱动的采样指令预测层 (无需 Key 参与)
        #    - offset_proj : 预测每个采样点的 (dx, dy) 像素偏移量
        #                   输出维度: num_heads * num_levels * num_points * 2
        #    - weight_proj : 预测每个采样点的注意力权重 (logits)
        #                   输出维度: num_heads * num_levels * num_points
        # ------------------------------------------------------------
        self.offset_proj = nn.Linear(embed_dim, num_heads * num_levels * num_points * 2)
        self.weight_proj = nn.Linear(embed_dim, num_heads * num_levels * num_points)

        # ------------------------------------------------------------
        # 3. 输出投影层 (多头融合后的最终线性变换)
        # ------------------------------------------------------------
        self.output_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

        # 执行特定的权重初始化策略 (偏移量强制归零是关键)
        self._init_weights()

    def _init_weights(self):
        """
        参数初始化策略 (这是可变形注意力收敛的关键技巧)

        1. value_proj / weight_proj / output_proj: 使用 Xavier 均匀初始化，正常学习。
        2. offset_proj: 权重视为 0，偏置为 0。
           为什么？训练初期，让采样点严格落在参考点 (reference point) 上，
           即初始行为等价于"只采样参考点位置的特征"，这能让模型先从标准注意力起步，
           再慢慢学习偏移，避免早期梯度崩塌或随机游走导致的不收敛。
        """
        nn.init.xavier_uniform_(self.value_proj.weight)
        nn.init.constant_(self.value_proj.bias, 0.0)

        # 偏移量投影层必须归零！这是无数实验验证过的"保命"技巧
        nn.init.constant_(self.offset_proj.weight, 0.0)
        nn.init.constant_(self.offset_proj.bias, 0.0)

        nn.init.xavier_uniform_(self.weight_proj.weight)
        nn.init.constant_(self.weight_proj.bias, 0.0)

        nn.init.xavier_uniform_(self.output_proj.weight)
        nn.init.constant_(self.output_proj.bias, 0.0)

    def forward(
        self,
        query: torch.Tensor,               # (B, Nq, C) 查询向量
        value_list: List[torch.Tensor],    # 多尺度特征图列表
        reference_points: torch.Tensor,    # (B, Nq, 2) 或 (B, Nq, L, 2)
        already_projected: bool = False,   # True表示value_list已经是多头格式
        spatial_shapes: Optional[List[tuple]] = None,  # 每个尺度的(H, W)，用于归一化
    ) -> torch.Tensor:
        """
        前向传播函数

        参数:
            query: (B, Nq, C)  - Nq 是 Query 的数量 (如目标查询个数或网格点数)
            value_list: 列表，长度为 L。每个元素是 (B, H_l, W_l, C) 或 (B, num_heads, head_dim, H_l, W_l)
            reference_points: 归一化参考点 [0,1]。
                              形状为 (B, Nq, 2) 时，所有尺度共用同一组参考点；
                              形状为 (B, Nq, L, 2) 时，每个尺度有独立的参考点 (更灵活)。
            already_projected: 性能优化开关。如果为 True，跳过内部的 value_proj 和 reshape。
            spatial_shapes: 可选，提供每个尺度的 (H_l, W_l) 元组列表，用于校验。

        返回:
            updated_query: (B, Nq, C)  聚合了多尺度采样特征后的更新 Query
        """
        B, Nq, _ = query.shape          # B: Batch大小, Nq: Query数量
        L = self.num_levels             # L: 尺度数量

        # 校验：传入的特征图数量必须等于设置的 num_levels
        if len(value_list) != L:
            raise ValueError(f"Expected {L} feature maps, got {len(value_list)}")
        
        # 如果没有提供 spatial_shapes，则从 value_list 中自动提取每个尺度的 (H, W)
        if spatial_shapes is None:
            spatial_shapes = [(v.shape[-2], v.shape[-1]) for v in value_list]

        # ================================================================
        # Step 1: 将 Value 特征图统一转换为多头格式
        # ================================================================
        if already_projected:
            # 情况 A：外部已经做好了投影和 reshape。
            # 此时 value_list 中每个元素形状应为 (B, num_heads, head_dim, H_l, W_l)
            # 直接使用，跳过本模块的 value_proj (省显存和省计算)
            multi_head_values = value_list
        else:
            # 情况 B：标准流程。输入是 (B, H_l, W_l, C)，需要在内部做投影。
            multi_head_values = []
            for v in value_list:  # v: (B, H_l, W_l, C)
                # 线性映射: (B, H, W, C) -> (B, H, W, C)
                v_proj = self.value_proj(v)
                
                # 拆分为多头: (B, H, W, num_heads, head_dim)
                v_proj = v_proj.view(
                    B, *v_proj.shape[1:3], self.num_heads, self.head_dim
                )
                # 调整为 grid_sample 期望的内存排列: (B, num_heads, head_dim, H, W)
                # 这相当于把多头信息提到前面，方便后续批量并行处理
                v_proj = v_proj.permute(0, 3, 4, 1, 2).contiguous()
                multi_head_values.append(v_proj)

        # ================================================================
        # Step 2: 由 Query 预测采样指令 (偏移量与权重)
        # ================================================================
        # 2.1 预测偏移量 offsets
        # (B, Nq, C) -> (B, Nq, num_heads * L * num_points * 2)
        offsets_raw = self.offset_proj(query)
        # 解构为方便索引的维度: (B, Nq, num_heads, L, num_points, 2)
        # 最后2维对应 (dx, dy)，是像素尺度上的偏移量 (例如 2.3 像素)
        offsets = offsets_raw.view(
            B, Nq, self.num_heads, L, self.num_points, 2
        )

        # 2.2 预测注意力权重 weights (原始 logits)
        # (B, Nq, C) -> (B, Nq, num_heads * L * num_points)
        weights_raw = self.weight_proj(query)
        # 先 reshape 为 (B, Nq, num_heads, L * num_points)
        weights = weights_raw.view(B, Nq, self.num_heads, L * self.num_points)
        # 对最后一个维度 (L * num_points) 做 Softmax
        # 物理含义：同一个 Query 的同一个 Head 内部，所有尺度 + 所有采样点的权重之和为 1
        weights = F.softmax(weights, dim=-1)
        # 再 reshape 为 (B, Nq, num_heads, L, num_points)，方便按尺度索引
        weights = weights.view(B, Nq, self.num_heads, L, self.num_points)

        # ================================================================
        # Step 3: 处理参考点 (Reference Points) 的广播逻辑
        # ================================================================
        # 若 reference_points 是 (B, Nq, 2)，则所有尺度共享同一组参考点。
        # 若 reference_points 是 (B, Nq, L, 2)，则每个尺度有独立的参考点。
        # 这里我们在维度上插入两个长度为1的维度，以适配后面的广播。
        if reference_points.dim() == 3:  # 形状: (B, Nq, 2)
            # -> (B, Nq, 1, 1, 2)  后续将自动广播到 num_heads 和 num_points
            ref_pts = reference_points[:, :, None, None, :]
        else:  # 形状: (B, Nq, L, 2)
            # -> (B, Nq, 1, L, 1, 2)
            ref_pts = reference_points[:, :, None, :, None, :]

        # ================================================================
        # Step 4: 核心多尺度循环采样与聚合
        # ================================================================
        # 初始化累加器: 用于存放所有尺度的加权采样特征，形状 (B, Nq, num_heads, head_dim)
        aggregated = query.new_zeros((B, Nq, self.num_heads, self.head_dim))

        # 逐个尺度遍历 (L 个尺度)
        for l in range(L):
            # 4.1 获取当前尺度的多头特征图
            # feat_l: (B, num_heads, head_dim, H_l, W_l)
            feat_l = multi_head_values[l]
            h_l, w_l = spatial_shapes[l]  # 当前尺度的高和宽

            # 4.2 将特征图转换为 grid_sample 需要的格式
            # 合并 Batch 和 Head 维度: (B, num_heads, head_dim, H, W) -> (B*num_heads, head_dim, H, W)
            # 这样 grid_sample 就可以并行处理所有 Head，无需循环
            feat_l_for_grid = feat_l.view(
                B * self.num_heads, self.head_dim, h_l, w_l
            )

            # 4.3 提取当前尺度的参考点 (Reference Points)
            # 从 ref_pts 中取出第 l 个尺度
            # 如果 ref_pts 是 (B, Nq, 1, 1, 2)，则取索引 l 时依然广播
            if reference_points.dim() == 3:
                # 所有尺度共享 ref_pts: (B, Nq, 1, 1, 2)
                ref_l = reference_points[:, :, None, None, :]
            else:
                # 取出第 l 个尺度: (B, Nq, 1, 1, 2)
                # 通过 [:, :, None, l, None, :] 保证最后形状是 (B, Nq, 1, 1, 2)
                ref_l = reference_points[:, :, None, l, None, :]

            # 4.4 提取当前尺度的偏移量
            # offsets: (B, Nq, num_heads, L, num_points, 2) -> 取第 l 个尺度
            # 结果形状: (B, Nq, num_heads, num_points, 2)
            offsets_l = offsets[:, :, :, l, :, :]

            # 4.5 将偏移量从像素尺度归一化到 [0,1] 相对尺度
            # 为什么？因为 ref_l 是归一化坐标 (0~1)，偏移必须与其处于同一量纲才能相加。
            # 归一化公式: norm_offset = pixel_offset / (W_l or H_l)
            normalizer = query.new_tensor([w_l, h_l]).view(1, 1, 1, 1, 2)
            offsets_l_norm = offsets_l / normalizer

            # 4.6 计算最终采样位置 (归一化坐标，范围约在 [0,1] 之间)
            # ref_l 广播到 (B, Nq, num_heads, num_points, 2) 然后相加
            sampling_locs = ref_l + offsets_l_norm

            # 4.7 将 [0,1] 坐标映射到 grid_sample 要求的 [-1, 1] 坐标
            # 映射关系: grid = 2 * location - 1  (0->-1, 1->1)
            grid = sampling_locs * 2.0 - 1.0

            # 4.8 调整 grid 维度以匹配 grid_sample
            # 当前 grid: (B, Nq, num_heads, num_points, 2)
            # 需要调整为: (B * num_heads, Nq, num_points, 2)
            # 因为 grid_sample 要求输入形状为 (N, H_out, W_out, 2)
            # 我们将 Nq 视为 H_out，num_points 视为 W_out
            grid = grid.permute(0, 2, 1, 3, 4).contiguous()
            grid = grid.view(B * self.num_heads, Nq, self.num_points, 2)

            # 4.9 执行双线性插值采样
            # 输入: feat_l_for_grid (B*num_heads, head_dim, H_l, W_l)
            #       grid          (B*num_heads, Nq, num_points, 2)
            # 输出: sampled_l     (B*num_heads, head_dim, Nq, num_points)
            sampled_l = F.grid_sample(
                feat_l_for_grid,
                grid,
                mode="bilinear",       # 双线性插值，使偏移量可微
                padding_mode="zeros",  # 越界部分补 0
                align_corners=False,   # False 表示坐标对应像素区域中心
            )

            # 4.10 恢复维度并重排
            # (B*num_heads, head_dim, Nq, num_points) -> (B, num_heads, head_dim, Nq, num_points)
            sampled_l = sampled_l.view(
                B, self.num_heads, self.head_dim, Nq, self.num_points
            )
            # -> (B, Nq, num_heads, num_points, head_dim)
            # 这样调整后，最后一维是特征，方便与权重做加权求和
            sampled_l = sampled_l.permute(0, 3, 1, 4, 2).contiguous()

            # 4.11 提取当前尺度的权重并扩维
            # weights: (B, Nq, num_heads, L, num_points) -> 取第 l 个尺度
            # 并添加最后一维: (B, Nq, num_heads, num_points, 1)
            w_l = weights[:, :, :, l, :].unsqueeze(-1)

            # 4.12 加权累加 (对 num_points 维度求和)
            # (sampled_l * w_l) 逐元素相乘，然后 .sum(dim=3) 消去 num_points 维
            # 结果累加到 aggregated 中
            # 物理含义：对于当前尺度，把 K 个采样点的特征按权重融合成一个向量
            aggregated = aggregated + (sampled_l * w_l).sum(dim=3)

        # ================================================================
        # Step 5: 多头融合与输出投影
        # ================================================================
        # aggregated: (B, Nq, num_heads, head_dim) -> 拼接成 (B, Nq, embed_dim)
        output = aggregated.view(B, Nq, self.embed_dim)
        # 输出投影层 (融合多头信息)
        output = self.output_proj(output)
        # Dropout 正则
        output = self.dropout(output)

        return output


In [ ]:

# ============================================================================
# 第二部分：BEV 专用融合模块 (显式复用上面的通用注意力)
# ============================================================================
class DeformableBEVFusion(nn.Module):
    """
    基于 Query 的多尺度可变形 BEV (鸟瞰图) 特征融合模块

    这个模块专为自动驾驶 BEV 感知设计，核心功能是：
    1. 接收一个 BEV 特征图 (B, C, H, W) 和若干目标 Query (B, Nq, C)。
    2. Query 驱动可变形采样，从 BEV 的多尺度金字塔中提取关键信息。
    3. 将更新后的 Query 全局语义“回注”到原始 BEV 特征图中，增强全图表达。

    设计亮点：
    - 极致性能：先对 BEV 做一次 Value 投影，再池化生成金字塔，避免重复计算。
    - 显式复用：所有采样逻辑完全委托给 MultiScaleDeformableAttention，代码干净。
    - 上下文回注：用 Query 均值作为全局上下文，与 BEV 每个像素相加，类似全局残差。
    """
    def __init__(
        self,
        embed_dim: int = 256,
        num_heads: int = 8,
        num_levels: int = 3,   # 通常使用 3 层金字塔: 1x, 1/2x, 1/4x
        num_points: int = 4,
        dropout: float = 0.1,
    ):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.num_levels = num_levels
        self.num_points = num_points
        self.head_dim = embed_dim // num_heads

        # ------------------------------------------------------------
        # 1. 直接复用通用的多尺度可变形注意力模块
        #    注意：我们会利用 already_projected=True 来跳过内部 Value 投影，
        #    因为 BEV 模块将在外部做一次统一的 Value 投影，效率更高。
        # ------------------------------------------------------------
        self.deform_attn = MultiScaleDeformableAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            num_points=num_points,
            num_levels=num_levels,
            dropout=dropout,
        )

        # ------------------------------------------------------------
        # 2. BEV 专用预投影层 (在池化前做，全图只做一次)
        #    这是 BEV 融合的性能核心：避免在每个尺度上重复做高维线性变换。
        # ------------------------------------------------------------
        self.value_proj = nn.Linear(embed_dim, embed_dim)

        # 3. 参考点预测层：让 Query 预测每个尺度独立的参考中心点
        #    输出维度 L*2，分别对应 L 个尺度的 (x, y)
        #    这样每个 Query 在不同尺度上的锚点可以不同，适应性更强。
        self.reference_points_proj = nn.Linear(embed_dim, num_levels * 2)

        # 4. 上下文回注融合网络 (MLP)
        #    将 Query 聚合信息经过非线性变换后，加到 BEV 特征图上。
        self.context_fusion = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(inplace=True),
            nn.Linear(embed_dim, embed_dim),
        )
        # 用于更新 Query 的残差归一化
        self.norm = nn.LayerNorm(embed_dim)

        self._init_weights()

    def _init_weights(self):
        """BEV 模块自身的初始化，注意参考点投影层使用 Xavier，偏移量初始化由内部类负责"""
        nn.init.xavier_uniform_(self.value_proj.weight)
        nn.init.constant_(self.value_proj.bias, 0.0)
        nn.init.xavier_uniform_(self.reference_points_proj.weight)
        nn.init.constant_(self.reference_points_proj.bias, 0.0)
        # 注意：偏移量 (offset_proj) 和权重 (weight_proj) 已经在
        # MultiScaleDeformableAttention 内部初始化为 0 了，这里无需再操作。

    def _build_projected_multi_scale(self, bev_feature: torch.Tensor):
        """
        BEV 特色优化：先对全图做一次 Value 投影，再平均池化生成多尺度金字塔。

        普通做法 (低效)：
            对每个尺度分别: (B,C,H,W) -> Linear -> Reshape。
            如果有 L=3 个尺度，线性变换要算 3 次，且中间有大量重复计算。

        高效做法 (本实现)：
            1. 将 BEV 展平为 (B, H*W, C)，做一次 Linear。
            2. 还原为 (B, C, H, W)。
            3. 分别用 avg_pool2d 得到 1x, 1/2x, 1/4x。
            4. 将每个尺度的特征转为多头格式 (B, num_heads, head_dim, H_l, W_l)。
            
        输入:
            bev_feature: (B, C, H, W)
        返回:
            multi_scale_list: List，每个元素 (B, num_heads, head_dim, H_l, W_l)
        """
        B, C, H, W = bev_feature.shape

        # Step 1: 展平并全局投影 (全图 Linear 只算一次)
        # (B, C, H, W) -> (B, C, H*W) -> (B, H*W, C)
        tokens = bev_feature.flatten(2).transpose(1, 2)
        # 线性映射: (B, H*W, C) -> (B, H*W, C)
        tokens_proj = self.value_proj(tokens)

        # Step 2: 还原为 2D 形状
        # (B, H*W, C) -> (B, C, H*W) -> (B, C, H, W)
        feat_proj = tokens_proj.transpose(1, 2).view(B, C, H, W).contiguous()

        # Step 3: 平均池化构建金字塔，并转为多头格式
        multi_scale_list = []
        for level in range(self.num_levels):
            # 计算当前尺度下采样倍数: scale = 2^level
            scale = 2 ** level
            if scale == 1:
                feat = feat_proj  # 原图尺寸
            else:
                # 使用 avg_pool2d 进行下采样，kernel_size=scale, stride=scale
                # 这模拟了 FPN 的降采样效果，保留低频信息
                feat = F.avg_pool2d(feat_proj, kernel_size=scale, stride=scale)

            B, C_l, H_l, W_l = feat.shape

            # 转换为多头格式: (B, C_l, H_l, W_l) -> (B, num_heads, head_dim, H_l, W_l)
            # 注意这里 C_l == embed_dim 恒成立
            feat = feat.view(B, self.num_heads, self.head_dim, H_l, W_l)
            multi_scale_list.append(feat)

        return multi_scale_list

    def forward(self, query: torch.Tensor, bev_feature: torch.Tensor):
        """
        前向传播

        输入:
            query: (B, Nq, C)          - 查询向量，可以是目标查询、网格查询或运动特征。
            bev_feature: (B, C, H, W)  - 原始 BEV 特征图 (来自图像或 LiDAR 分支)

        输出:
            fused_bev: (B, C, H, W)    - 融合了 Query 全局语义的增强 BEV 特征图。
                                        空间尺寸不变，但每个像素都获得了全局上下文信息。
        """
        B, Nq, C = query.shape
        _, _, H, W = bev_feature.shape

        # ================================================================
        # Stage 1: BEV 预投影 + 多尺度金字塔构建 (性能优化)
        # ================================================================
        # value_list 中每个元素形状: (B, num_heads, head_dim, H_l, W_l)
        # 这些特征已经经过了 value_proj 映射，且分好了多头，可以直接送入通用模块。
        value_list = self._build_projected_multi_scale(bev_feature)

        # ================================================================
        # Stage 2: 预测每个 Query 在不同尺度上的参考点 (Anchor)
        # ================================================================
        # 从 Query 预测原始 logits: (B, Nq, C) -> (B, Nq, L*2)
        ref_pts_raw = self.reference_points_proj(query)
        # 经过 Sigmoid 将坐标限制在 [0, 1] 区间，作为归一化参考点
        # 形状: (B, Nq, L, 2) -> 每一行代表 (x, y) 归一化坐标
        reference_points = torch.sigmoid(ref_pts_raw).view(B, Nq, self.num_levels, 2)

        # ================================================================
        # Stage 3: 调用通用可变形注意力 (显式复用核心逻辑)
        # ================================================================
        # 关键参数 already_projected=True:
        #   告诉通用模块跳过内部 value_proj，直接使用我们传进去的多头特征。
        # 返回: updated_query (B, Nq, C)
        # 该 Query 已经聚合了 BEV 多尺度上的关键形变特征。
        updated_query = self.deform_attn(
            query=query,
            value_list=value_list,
            reference_points=reference_points,
            already_projected=True,   # 跳过内部投影，使用外部预投影特征
        )

        # ================================================================
        # Stage 4: BEV 上下文回注 (Context Injection)
        # ================================================================
        # 这是 BEV 融合模块区别于纯注意力模块的额外功能。
        # 目标：将 Query 中编码的"全局高阶语义"分发到 BEV 特征图的每一个像素上。

        # 4.1 对所有 Query 取均值，得到全局上下文向量
        # (B, Nq, C) -> (B, 1, C)
        global_context = updated_query.mean(dim=1, keepdim=True)

        # 4.2 通过 MLP 增强表达能力
        global_context = self.context_fusion(global_context)  # (B, 1, C)

        # 4.3 展平 BEV 特征图
        # (B, C, H, W) -> (B, C, H*W) -> (B, H*W, C)
        bev_tokens = bev_feature.flatten(2).transpose(1, 2)

        # 4.4 广播加法：将全局上下文加到每个像素的特征上
        # (B, H*W, C) + (B, 1, C) -> (B, H*W, C)
        # 这相当于对 BEV 的每个位置做了一个全局残差连接，引入了 Query 的意图信息。
        fused_tokens = bev_tokens + global_context

        # 4.5 还原为 2D 图像格式
        # (B, H*W, C) -> (B, C, H*W) -> (B, C, H, W)
        fused_bev = fused_tokens.transpose(1, 2).view(B, C, H, W).contiguous()

        return fused_bev